## Setup
This notebook uses the local `.venv` virtual environment.  
**Select kernel:** `Python (Spradley)` in the top-right kernel picker before running.

In [ ]:
# ── C0: Config ───────────────────────────────────────────────────────────────
# Edit the values below before running. These override pipeline_core.py defaults.
# If you skip this cell, pipeline_core.py defaults apply automatically.

import importlib
import pipeline_core
importlib.reload(pipeline_core)  # force fresh load after any pipeline_core.py change

# ── Input data ───────────────────────────────────────────────────────────────
# INPUT_FILE: filename inside interview_input/
# INPUT_FORMAT: "spradley_v2" for Spradley platform exports
#
# NOTE (Roskilde branch): C0c pulls data directly from Supabase and sets
# `interviews` without reading this CSV. INPUT_FILE is used only if C3 is
# run instead of C0c (e.g. for a resume run from a checkpoint).
pipeline_core.CONFIG["INPUT_FILE"]   = "roskilde-festival-2026.csv"
pipeline_core.CONFIG["INPUT_FORMAT"] = "spradley_v2"

# ── Participant inclusion (Roskilde) ─────────────────────────────────────────
# MIN_ANSWERS: minimum valid Q&A rows per session to include. 2 = anyone who
# answered at least the first two turns. Set to 6 to include completed only.
pipeline_core.CONFIG["MIN_ANSWERS"]           = 2
pipeline_core.CONFIG["FILTER_EMPTY_QUESTIONS"] = True   # drops app-bug rows (blank question_asked)

# ── Tunable per run -- edit here ─────────────────────────────────────────────
pipeline_core.CONFIG["LLM_MODEL"]       = "claude-haiku-4-5-20251001"
pipeline_core.CONFIG["LLM_TEMPERATURE"] = 0.2
pipeline_core.CONFIG["L2_CODES_RANGE"]  = (10, 20)  # codes per interview
pipeline_core.CONFIG["L3_CODES_RANGE"]  = (30, 60)  # consolidated codes across all respondents
pipeline_core.CONFIG["CLUSTERS_RANGE"]  = (4, 6)    # thematic clusters
pipeline_core.CONFIG["MAX_EXPERIMENTS"]  = 3

print("Config active:")
print(f"  Input:         {pipeline_core.CONFIG['INPUT_FILE']}  (format={pipeline_core.CONFIG['INPUT_FORMAT']})")
print(f"  MIN_ANSWERS:   {pipeline_core.CONFIG['MIN_ANSWERS']}")
print(f"  Model:         {pipeline_core.CONFIG['LLM_MODEL']}  (temp={pipeline_core.CONFIG['LLM_TEMPERATURE']})")
print(f"  L2 range:      {pipeline_core.CONFIG['L2_CODES_RANGE']}")
print(f"  L3 range:      {pipeline_core.CONFIG['L3_CODES_RANGE']}")
print(f"  Clusters:      {pipeline_core.CONFIG['CLUSTERS_RANGE']}")
print(f"  Experiments:   {pipeline_core.CONFIG['MAX_EXPERIMENTS']}")

In [ ]:
# ── C0b: Supabase data pull + Vertex Haiku test ──────────────────────────────
# Optional -- skip when using a local CSV already in interview_input/.
# Step 1: leave INTERVIEW_ID = None to discover recent interviews.
# Step 2: set INTERVIEW_ID to the UUID you want, re-run to pull and save as CSV.

import os, httpx, json
from dotenv import load_dotenv
import pipeline_core

load_dotenv(pipeline_core.KEYS_ENV)

SUPABASE_URL = os.environ["SUPABASE_URL"]
SUPABASE_KEY = os.environ["SUPABASE_SERVICE_ROLE_KEY"]
HEADERS = {
    "apikey":        SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Accept":        "application/json",
}

# ── Set INTERVIEW_ID to pull a specific interview, or leave None to discover ──
INTERVIEW_ID = None   # e.g. "abc123de-..."

print("=" * 60)
print("PART A: Supabase")
print("=" * 60)

def sb_get(path, params=None):
    with httpx.Client(verify=False) as c:
        r = c.get(f"{SUPABASE_URL}/rest/v1/{path}", headers=HEADERS, params=params)
        r.raise_for_status()
        return r.json()

if INTERVIEW_ID is None:
    # ── Discovery: list recent interviews ─────────────────────────────────────
    rows = sb_get("interviews", {
        "select": "id,title,status,sent_at,created_at",
        "order":  "created_at.desc",
        "limit":  "20",
    })
    print(f"Recent interviews (newest first):\n")
    for iv in rows:
        print(f"  {iv['id']}")
        print(f"    title:    {iv['title']}")
        print(f"    status:   {iv['status']}  |  sent: {(iv.get('sent_at') or '')[:10]}  |  created: {iv.get('created_at','')[:10]}")
    print("\nSet INTERVIEW_ID to the UUID above and re-run to pull the full dataset.")

else:
    # ── Pull: join interviews + assignments + answers + questions ─────────────
    import pandas as pd

    # 1. Get interview metadata
    iv_rows = sb_get("interviews", {"select": "id,title", "id": f"eq.{INTERVIEW_ID}"})
    iv_title = iv_rows[0]["title"] if iv_rows else INTERVIEW_ID
    print(f"Interview: {iv_title!r}  ({INTERVIEW_ID})")

    # 2. Get all assignments for this interview
    assignments = sb_get("interview_assignments", {
        "select": "id,employee_id,status,is_test",
        "interview_id": f"eq.{INTERVIEW_ID}",
    })
    print(f"Assignments: {len(assignments)}")
    asgn_ids = [a["id"] for a in assignments]
    asgn_map = {a["id"]: a for a in assignments}

    # 3. Get all answers for these assignments (with question text via FK embed)
    answers = sb_get("interview_answers", {
        "select":        "assignment_id,question_id,answer_text,is_skipped,turn_index",
        "assignment_id": f"in.({','.join(asgn_ids)})",
        "limit":         "10000",
    })
    print(f"Answers:     {len(answers)}")

    # 4. Get question texts for this interview
    questions = sb_get("questions", {
        "select":       "id,text,order_index",
        "interview_id": f"eq.{INTERVIEW_ID}",
        "order":        "order_index.asc",
    })
    q_text = {q["id"]: q["text"] for q in questions}
    print(f"Questions:   {len(questions)}")

    # 5. Build spradley_v2 DataFrame
    rows = []
    for ans in answers:
        asgn = asgn_map.get(ans["assignment_id"], {})
        rows.append({
            "employee_id":        asgn.get("employee_id", ans["assignment_id"]),
            "thread":             iv_title,
            "turn_index":         ans["turn_index"],
            "question_asked":     q_text.get(ans["question_id"], ""),
            "answer_text":        ans["answer_text"] or "",
            "is_skipped":         ans["is_skipped"],
            "participant_status": asgn.get("status", ""),
            "is_test":            asgn.get("is_test", False),
        })

    df = pd.DataFrame(rows).sort_values(["employee_id", "turn_index"])
    print(f"\nRows in output: {len(df)}")
    print(f"Columns: {list(df.columns)}")
    print(f"Completed respondents: {df[df['participant_status']=='completed']['employee_id'].nunique()}")

    # 6. Save and wire into pipeline CONFIG
    safe_title = iv_title.lower().replace(" ", "-").replace("/", "-")[:40]
    os.makedirs(pipeline_core.INPUT_DIR, exist_ok=True)
    out_path = os.path.join(pipeline_core.INPUT_DIR, f"{safe_title}.csv")
    df.to_csv(out_path, index=False)
    print(f"\nSaved: {out_path}")

    pipeline_core.CONFIG["INPUT_FILE"]   = f"{safe_title}.csv"
    pipeline_core.CONFIG["INPUT_FORMAT"] = "spradley_v2"
    print(f"CONFIG updated -> INPUT_FILE={safe_title}.csv")
    print("Run C1 -> C3 next (skip Resume cell -- this is fresh data).")

# ── Part B: Vertex AI Haiku test ──────────────────────────────────────────────
print("\n" + "=" * 60)
print("PART B: Vertex AI Haiku test")
print("=" * 60)

try:
    import google.auth
    from anthropic import AnthropicVertex

    gcp_project  = os.environ.get("GCP_PROJECT_ID", "spradley-ai")
    gcp_location = os.environ.get("GCP_LOCATION",   "europe-west1")

    print(f"Project: {gcp_project}  |  Region: {gcp_location}")
    client = AnthropicVertex(project_id=gcp_project, region=gcp_location)
    msg = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=20,
        messages=[{"role": "user", "content": "Respond with just: OK"}],
    )
    print(f"Vertex Haiku WORKS: {msg.content[0].text!r}")

except Exception as e:
    print(f"Vertex Haiku FAILED: {type(e).__name__}: {e}")
    if "credentials" in str(e).lower() or "DefaultCredentials" in type(e).__name__:
        print("-> Ask colleague for GCP_CREDENTIALS_JSON (service account used on Vercel).")
        print("   Save as a .json file and add to keys.env:")
        print("   GOOGLE_APPLICATION_CREDENTIALS=C:\\path\\to\\spradley-sa.json")
    elif "404" in str(e):
        print("-> 404: Haiku not available in europe-west1.")

In [ ]:
# ── C0c: Roskilde Festival data pull ─────────────────────────────────────────
# ROSKILDE-SPECIFIC -- lives on the roskilde fork, not on main.
# Source: spradley-dev / research_sessions + research_answers tables
# Project: "Festival waste Roskilde 2026"
# Outputs: `interviews` list in pipeline format + Q&A preview.
#
# Inclusion rule: sessions with >= MIN_ANSWERS valid Q&A rows (default 2).
# Exclusion rule: rows with blank question_asked (app UI bugs) are dropped.

import os, httpx
from dotenv import load_dotenv
import pipeline_core

load_dotenv(pipeline_core.KEYS_ENV)

ACCESS_TOKEN = os.environ["SUPABASE_ACCESS_TOKEN"]
DEV_REF      = "jbifjmcsausdhyfydrnu"
PROJECT_ID   = "c6b21bb1-01b4-4878-8b0d-0e2cdddccda0"
MIN_ANSWERS  = pipeline_core.CONFIG.get("MIN_ANSWERS", 2)

def sql(query):
    with httpx.Client(verify=False, timeout=60.0) as c:
        r = c.post(
            f"https://api.supabase.com/v1/projects/{DEV_REF}/database/query",
            headers={"Authorization": f"Bearer {ACCESS_TOKEN}", "Content-Type": "application/json"},
            json={"query": query},
        )
        r.raise_for_status()
        return r.json()

# ── 1. Pull all sessions (no status filter) ───────────────────────────────────
rows = sql(f"""
    SELECT
        ra.session_id::text              AS interview_id,
        ra.turn_index::text              AS turn_number,
        COALESCE(ra.ai_question_text,
                 rq.text, '')            AS question,
        COALESCE(ra.answer_text, '')     AS answer,
        ra.is_skipped,
        rs.status                        AS participant_status
    FROM research_answers ra
    JOIN research_sessions rs ON rs.id = ra.session_id
    LEFT JOIN research_questions rq ON rq.id = ra.question_id
    WHERE rs.project_id = '{PROJECT_ID}'
    ORDER BY ra.session_id, ra.turn_index;
""")

print(f"Raw rows from DB: {len(rows)}")
print(f"Unique sessions:  {len(set(r['interview_id'] for r in rows))}")

# ── 2. Row-level filtering: skip flagged / empty answer / blank question ───────
from collections import defaultdict

by_session = defaultdict(list)
for r in rows:
    if r["is_skipped"]:
        continue
    if not r["answer"].strip():
        continue
    if not r["question"].strip():          # FILTER_EMPTY_QUESTIONS
        continue
    by_session[r["interview_id"]].append({
        "turn_number":        r["turn_number"],
        "question":           r["question"],
        "answer":             r["answer"],
        "participant_status": r["participant_status"],
    })

# ── 3. Session-level filter: MIN_ANSWERS ─────────────────────────────────────
interviews = []
dropped_sessions = []
for sid, pairs in by_session.items():
    if len(pairs) >= MIN_ANSWERS:
        interviews.append({
            "interview_id": sid,
            "qa_pairs":     sorted(pairs, key=lambda x: int(x["turn_number"])),
        })
    else:
        dropped_sessions.append((sid, len(pairs)))

status_counts = defaultdict(int)
for iv in interviews:
    status_counts[iv["qa_pairs"][0]["participant_status"]] += 1

print(f"\nSessions after filters (MIN_ANSWERS={MIN_ANSWERS}): {len(interviews)}")
for status, n in sorted(status_counts.items()):
    print(f"  {status}: {n}")
print(f"Dropped (< {MIN_ANSWERS} answers): {len(dropped_sessions)}")
print(f"Total Q&A pairs kept: {sum(len(iv['qa_pairs']) for iv in interviews)}")

# ── 4. Q&A preview ───────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print(f" Q&A PREVIEW  (first 5 of {len(interviews)} respondents)")
print("=" * 65)

for i, iv in enumerate(interviews[:5], 1):
    print(f"\n[Respondent {i:02d}]  {iv['interview_id'][:8]}...")
    for qa in iv["qa_pairs"]:
        q_short = qa["question"][:80] + ("..." if len(qa["question"]) > 80 else "")
        print(f"  t{qa['turn_number']}  Q: {q_short}")
        a_lines = qa["answer"].replace("\n", " ")
        print(f"      A: {a_lines[:120]}{'...' if len(a_lines) > 120 else ''}")

print("\n" + "=" * 65)
print(f"interviews is ready ({len(interviews)} sessions) -- run C1 -> C5 onwards.")
print("Note: C3 (load_interviews) is bypassed; interviews is set directly here.")

In [ ]:
# ── C1: Imports & env ────────────────────────────────────────────────────────
import os
import json
from collections import defaultdict
from dotenv import load_dotenv
import pipeline_core

load_dotenv(pipeline_core.KEYS_ENV)

loaded_keys = [k for k in [
    "ANTHROPIC_API_KEY", "SUPABASE_URL", "GCP_PROJECT_ID"
] if os.getenv(k)]
print(f"Env loaded from '{pipeline_core.KEYS_ENV}'")
print(f"Keys found: {loaded_keys if loaded_keys else 'none -- check keys.env'}")

In [ ]:
# ── Resume: Load checkpoint ───────────────────────────────────────────────────
# Run this instead of C3-C6 when pipeline_output/ already exists from a prior run.
#
# Re-run sequence:  C0 -> C1 -> this cell -> C8b -> C9 -> C9b -> C11
#
# This cell loads the INPUTS for C8b (interview_store, db, global_store with l3_codes).
# It does NOT load clusters/lineage -- those are rebuilt by C8b with the current CLUSTERS_RANGE.
# Run C8 (L3 consolidation) only if you want to recode from scratch.
# Run C9c (dimension scoring) only if you also run C3 (needs raw interview objects).

import os, json as _json

def _chk(name):
    path = os.path.join(pipeline_core.OUTPUT_DIR, f"{name}.json")
    with open(path, encoding="utf-8") as f:
        return _json.load(f)

db              = _chk("db")
interview_store = _chk("interview_store")
global_store    = _chk("global_store")  # contains l3_codes needed by C8b
run_meta        = _chk("run_meta")
n_interviews    = run_meta["n_interviews"]

print(f"db: {len(db)} entries  |  interview_store: {len(interview_store)} interviews")
print(f"l3_codes in global_store: {len(global_store.get('l3_codes', []))}")
print(f"n_interviews: {n_interviews}")
print("\nRun C8b next to re-cluster with CLUSTERS_RANGE =", pipeline_core.CONFIG["CLUSTERS_RANGE"])

In [28]:
# ── C2: LLM client ───────────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C2" there to find/edit the implementation).
# To swap providers: edit pipeline_core.CONFIG["LLM_PROVIDER"] in C0 above.
# verify=False   — bypasses corporate TLS inspection (API key still authenticates).
# trust_env=False — ignores any HTTPS_PROXY / HTTP_PROXY env vars; connect directly.

from pipeline_core import call_llm

cfg = pipeline_core.CONFIG
print(f"LLM client ready: {cfg['LLM_PROVIDER']} / {cfg['LLM_MODEL']}  (temperature={cfg['LLM_TEMPERATURE']})")

LLM client ready: anthropic / claude-haiku-4-5-20251001  (temperature=0.2)


In [ ]:
# ── C3: Data ingestion ───────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C3"). Edit there to adapt to future input formats.

from pipeline_core import load_interviews

interviews = load_interviews()

print(f"Loaded {len(interviews)} interviews\n")
for iv in interviews:
    print(f"  {iv['interview_id'][:8]}...  {len(iv['qa_pairs'])} Q&A pairs")

print("\nSample (first interview, first 2 pairs):")
for qa in interviews[0]["qa_pairs"][:2]:
    print(f"  [t{qa['turn_number']}] Q: {qa['question'][:65]}")
    print(f"         A: {qa['answer'][:65]}")
    print()

In [30]:
# ── C4: Anonymizer ───────────────────────────────────────────────────────────
# Defined in pipeline_core.py (search "C4"). PII patterns are editable there.

from pipeline_core import anonymize

sample_raw  = interviews[0]["qa_pairs"][0]["answer"]
sample_anon = anonymize(sample_raw)
print("Anonymizer ready.")
print(f"  Before: {sample_raw[:80]}")
print(f"  After:  {sample_anon[:80]}")

Anonymizer ready.
  Before: Great team and exciting new things coming up!
  After:  Great team and exciting new things coming up!


In [31]:
# ── C5: DB init (L1) ─────────────────────────────────────────────────────────
# Builds the primary Q&A fact table. PK = interview_question_id (e.g. "6abc1e3b_t1").
# Shape is fixed after this cell; downstream cells read but do not modify db.

db = {}
for iv in interviews:
    prefix = iv["interview_id"][:8]
    for qa in iv["qa_pairs"]:
        iq_id = f"{prefix}_t{qa['turn_number']}"
        db[iq_id] = {
            "interview_id":      iv["interview_id"],
            "turn_number":       qa["turn_number"],
            "question":          qa["question"],
            "anonymised_answer": anonymize(qa["answer"]),
        }

print(f"DB initialised: {len(db)} entries")
sample_key = next(iter(db))
print(f"\nSample entry  ({sample_key}):")
for k, v in db[sample_key].items():
    print(f"  {k}: {str(v)[:80]!r}")

DB initialised: 49 entries

Sample entry  (6abc1e3b_t1):
  interview_id: '6abc1e3b-0f8a-423d-aed6-6b84640dcda1'
  turn_number: '1'
  question: '[initial mood opener]'
  anonymised_answer: 'Great team and exciting new things coming up!'


In [ ]:
# ── C6: L2 per-interview coding ───────────────────────────────────────────────
# One LLM call per interview. The model sees all Q&A turns in sequence and returns
# 20-30 thematic codes with the turn IDs that support each code.
# Prompt defined in pipeline_core.py (search "C6").

from pipeline_core import code_one_interview

interview_store = {}

by_interview = defaultdict(list)
for iq_id, entry in db.items():
    by_interview[entry["interview_id"]].append((iq_id, entry))

for interview_id, id_entry_pairs in by_interview.items():
    prefix = interview_id[:8]
    qa_entries = [
        (iq_id, e["question"], e["anonymised_answer"])
        for iq_id, e in sorted(id_entry_pairs, key=lambda x: int(x[1]["turn_number"]))
    ]
    l2_codes = code_one_interview(prefix, qa_entries)
    interview_store[interview_id] = {"l2_codes": l2_codes}
    print(f"  {prefix}:  {len(l2_codes)} L2 codes")

print(f"\nL2 coding done: {len(interview_store)} interviews")
sample_iv = next(iter(interview_store))
print(f"\nSample L2 codes ({sample_iv[:8]}...):")
for item in interview_store[sample_iv]["l2_codes"][:3]:
    print(f"  {item['code']!r:42}  <- {item['source_qa_ids']}")

In [ ]:
# ── C8: Global Consolidator (L3) ─────────────────────────────────────────────
# All L2 codes → L3_CODES_RANGE consolidated codes with merge lineage.
# Prompt defined in pipeline_core.py (search "C8").

from pipeline_core import consolidate_l3

n_interviews = len(interview_store)
all_l2_codes = [item["code"] for store in interview_store.values() for item in store["l2_codes"]]

global_store = {"l3_codes": consolidate_l3(all_l2_codes, n_interviews)}

n_l3 = len(global_store["l3_codes"])
l3_range = pipeline_core.CONFIG["L3_CODES_RANGE"]
print(f"L3 consolidation done: {n_l3} codes  (range: {l3_range[0]}-{l3_range[1]})")
print("\nFirst 10 L3 codes:")
for item in global_store["l3_codes"][:10]:
    print(f"  {item['code']!r:42}  <- {item['merged_from_l2']}")

In [ ]:
# ── C8b: Theme Clustering + Lineage ──────────────────────────────────────────
# L3 codes → named clusters. Prompt defined in pipeline_core.py (search "C8").
# Lineage (PK = cluster name) is built in this cell — it is orchestration, not a function.

from pipeline_core import cluster_l3_codes

l3_list     = [item["code"] for item in global_store["l3_codes"]]
cluster_map = cluster_l3_codes(l3_list)

# ── Build lookup tables ───────────────────────────────────────────────────────
l3_to_l2 = {item["code"]: item["merged_from_l2"] for item in global_store["l3_codes"]}

l2_to_interview = {}
l2_to_qa_ids    = {}
for interview_id, store in interview_store.items():
    for item in store["l2_codes"]:
        l2_to_interview[item["code"]] = interview_id
        l2_to_qa_ids[item["code"]]    = item["source_qa_ids"]

# ── Build lineage ─────────────────────────────────────────────────────────────
lineage  = {}
clusters = {}

for name, l3_codes_in_cluster in cluster_map.items():
    l2_entries, qa_ids = [], []
    for l3_code in l3_codes_in_cluster:
        for l2_code in l3_to_l2.get(l3_code, []):
            interview_id  = l2_to_interview.get(l2_code, "unknown")
            source_qa_ids = l2_to_qa_ids.get(l2_code, [])
            l2_entries.append({"code": l2_code, "interview_id": interview_id,
                               "source_qa_ids": source_qa_ids})
            qa_ids.extend(source_qa_ids)

    lineage[name] = {
        "l3_codes":  l3_codes_in_cluster,
        "l2_codes":  l2_entries,
        "l1_qa_ids": list(dict.fromkeys(qa_ids)),
    }
    clusters[name] = {"l3_codes": l3_codes_in_cluster, "story": ""}

cl_range = pipeline_core.CONFIG["CLUSTERS_RANGE"]
print(f"Clustering done: {len(clusters)} clusters (range: {cl_range[0]}-{cl_range[1]})\n")
for name, lin in lineage.items():
    print(f"  {name}")
    print(f"    Sources: {lin['l1_qa_ids'][:5]}{'...' if len(lin['l1_qa_ids']) > 5 else ''}")

In [ ]:
# ── C9: LLM Explainer ────────────────────────────────────────────────────────
# Per cluster: structured finding (category, tagline, summary, quotes, tag).
# Prompts defined in pipeline_core.py (search "C9").

from pipeline_core import explain_cluster, propose_experiments

for name, lin in lineage.items():
    iq_ids = lin["l1_qa_ids"]

    qa_lines = []
    for iq_id in iq_ids:
        if iq_id in db:
            e = db[iq_id]
            qa_lines.append(f"Q: {e['question']}\nA: {e['anonymised_answer']}")

    qa_pairs_text = "\n\n".join(qa_lines) if qa_lines else "(no source answers found)"
    result        = explain_cluster(name, clusters[name]["l3_codes"], qa_pairs_text)
    voice_count   = len({e["interview_id"] for e in lin["l2_codes"]})

    clusters[name].update({
        "category":    result.get("category", "ambiguous"),
        "tagline":     result.get("tagline", ""),
        "summary":     result.get("summary", ""),
        "quotes":      result.get("quotes", []),
        "tag":         result.get("tag", ""),
        "story":       result.get("summary", ""),
        "voice_count": voice_count,
    })
    print(f"[{result.get('category', '?')}]  {name}  ·  {result.get('tag', '')}  ·  {voice_count} voices")

# ── Experiment proposals ──────────────────────────────────────────────────────
# One batched call across all concern / ambiguous clusters. The LLM selects the
# MAX_EXPERIMENTS most impactful ones and returns source_cluster on each.
needs_attention = [
    (name, data) for name, data in clusters.items()
    if data.get("category") in ("concern", "ambiguous")
]
experiments = propose_experiments(needs_attention)

print(f"\nExperiment proposals: {len(experiments)}  (max={pipeline_core.CONFIG['MAX_EXPERIMENTS']})")
for exp in experiments:
    print(f"  · {exp['title']}  [{exp.get('source_cluster', '?')}]  ({exp.get('tag', '')})")

In [ ]:
# ── C9b: Headline narrative ───────────────────────────────────────────────────
# Generates the report opening from all cluster findings.
# Run after C9 (needs tagline + summary on every cluster).

from pipeline_core import generate_headline

print("Generating headline narrative...")
headline_data = generate_headline(
    clusters,
    n_interviews=len(interview_store),
    model=pipeline_core.CONFIG["LLM_MODEL"],
)
global_store["headline"]        = headline_data.get("headline", "")
global_store["note_to_protect"] = headline_data.get("note_to_protect", "")

print("\n--- Headline ---")
print(global_store["headline"])

In [ ]:
# ── C9c: Emotional dimension scoring ─────────────────────────────────────────
# One small LLM call per interview scores which of 6 emotional dimensions that
# speaker personally expressed (first-person only -- not reports about others).
# Separate from C6 to protect L2 coding quality.
# Writes dimension_store.json to pipeline_output/ for the Topics tab radar chart.

from pipeline_core import score_dimensions_one_interview, EMOTIONAL_DIMENSIONS
from collections import defaultdict as _defaultdict
import json, os as _os

# Reconstruct interview objects from db if running from Resume checkpoint (C3 not run)
if "interviews" not in dir() or not isinstance(interviews, list):
    _by_iv = _defaultdict(list)
    for _iq_id, _entry in db.items():
        _by_iv[_entry["interview_id"]].append({
            "turn_number": _entry["turn_number"],
            "question":    _entry["question"],
            "answer":      _entry["anonymised_answer"],
        })
    interviews = [
        {"interview_id": iv_id,
         "qa_pairs": sorted(pairs, key=lambda x: int(x["turn_number"]))}
        for iv_id, pairs in _by_iv.items()
    ]
    print(f"Reconstructed {len(interviews)} interview objects from db (anonymised answers).")

dimension_store: dict = {}
_total: dict = {d: 0 for d in EMOTIONAL_DIMENSIONS}
_total["interviews"] = 0

for _iv in interviews:
    _prefix = _iv["interview_id"][:8]
    _qa_entries = [
        (f"{_prefix}_t{qa['turn_number']}", qa["question"], qa["answer"])
        for qa in _iv["qa_pairs"]
    ]
    _scores = score_dimensions_one_interview(_prefix, _qa_entries)
    dimension_store[_iv["interview_id"]] = _scores
    for d in EMOTIONAL_DIMENSIONS:
        _total[d] += _scores.get(d, 0)
    _total["interviews"] += 1
    dimension_store["_total"] = dict(_total)   # snapshot after each interview
    print(f"  {_prefix}: {[d for d in EMOTIONAL_DIMENSIONS if _scores.get(d)]}")

_dim_path = _os.path.join(pipeline_core.OUTPUT_DIR, "dimension_store.json")
with open(_dim_path, "w", encoding="utf-8") as _f:
    json.dump(dimension_store, _f, indent=2)

_n = _total["interviews"]
print(f"\nDimension totals ({_n} interviews):")
for d in EMOTIONAL_DIMENSIONS:
    _pct = round(_total[d] / _n * 100) if _n else 0
    print(f"  {d}: {_total[d]} speakers  ({_pct}%)")
print(f"\nSaved: {_dim_path}")

In [ ]:
# ── C11: Persist ─────────────────────────────────────────────────────────────
# Write all data structures to OUTPUT_DIR as pretty-printed JSON.
# Re-run after any cell to checkpoint progress.

import datetime

os.makedirs(pipeline_core.OUTPUT_DIR, exist_ok=True)

to_save = {
    "db":              db,
    "interview_store": interview_store,
    "global_store":    global_store,
    "lineage":         lineage,
    "clusters":        clusters,
    "experiments":     globals().get("experiments", []),
}

for name, data in to_save.items():
    path = os.path.join(pipeline_core.OUTPUT_DIR, f"{name}.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    size_kb = os.path.getsize(path) / 1024
    print(f"  Saved: {path}  ({size_kb:.1f} KB)")

# ── run_meta.json — attributes results to this model run ──────────────────────
run_meta = {
    "model":        pipeline_core.CONFIG["LLM_MODEL"],
    "temperature":  pipeline_core.CONFIG["LLM_TEMPERATURE"],
    "timestamp":    datetime.datetime.utcnow().isoformat() + "Z",
    "n_interviews": len(interview_store),
    "input_file":   pipeline_core.CONFIG["INPUT_FILE"],
}
meta_path = os.path.join(pipeline_core.OUTPUT_DIR, "run_meta.json")
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(run_meta, f, indent=2)
print(f"  Saved: {meta_path}")

print(f"\nAll files written to '{pipeline_core.OUTPUT_DIR}'")

In [ ]:
# ── C12: Report renderer ─────────────────────────────────────────────────────
# Renders the full HR report as styled HTML directly in the notebook output.
# Implementation lives in pipeline_core.py (search "C12 / C13").

from IPython.display import display, HTML
from pipeline_core import build_inline_report_html

display(HTML(build_inline_report_html(clusters, interviews, experiments)))